In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
df = pd.read_csv("ahmedabad_aqi.csv")

In [3]:
df.shape

(1298, 12)

In [4]:
df.head()

,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi
0,2022-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-08-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-08-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-08-04,36.726316,22.131579,303.157895,22.810526,10.163158,59.473684,0.498947,11.052632,1.594737,NaN,NaN
4,2022-08-05,28.758333,19.912500,304.416667,23.770833,10.670833,55.166667,0.332083,1.208333,1.295833,70.947368,47.421053


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1298 entries, 0 to 1297
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1298 non-null   object 
 1   pm10                   1295 non-null   float64
 2   pm2_5                  1295 non-null   float64
 3   carbon_monoxide        1295 non-null   float64
 4   nitrogen_dioxide       1295 non-null   float64
 5   sulphur_dioxide        1295 non-null   float64
 6   ozone                  1295 non-null   float64
 7   aerosol_optical_depth  1295 non-null   float64
 8   dust                   1295 non-null   float64
 9   uv_index               1295 non-null   float64
 10  us_aqi                 1294 non-null   float64
 11  european_aqi           1294 non-null   float64
dtypes: float64(11), object(1)
memory usage: 121.8+ KB


In [7]:
df.describe()

,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi
count,1295.000000,1295.000000,1295.000000,1295.000000,1295.000000,1295.000000,1295.000000,1295.000000,1295.000000,1294.000000,1294.000000
mean,49.216314,30.760887,389.764569,20.242080,10.991214,68.783120,0.392560,22.073915,1.592543,90.757333,59.958272
std,21.658694,16.449835,196.050090,12.244387,5.763322,17.222297,0.213885,27.659935,0.563606,32.843696,19.075100
min,1.000000,0.704167,59.083333,0.000000,0.241667,26.208333,0.030833,0.000000,0.129167,15.375000,13.583333
25%,33.297917,17.708333,213.979167,10.927083,6.208333,54.666667,0.230833,2.604167,1.137500,63.635417,44.406250
50%,45.162500,26.604167,372.041667,16.745833,10.341667,69.125000,0.349167,11.416667,1.525000,83.083333,62.229167
75%,62.212500,40.179167,526.562500,26.379167,14.441667,81.687500,0.511042,31.187500,2.009375,112.562500,72.833333
max,154.337500,115.800000,1247.541667,80.608333,47.575000,119.666667,1.797083,296.000000,2.947917,182.083333,133.166667


In [8]:
print("Missing values:\n", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing values:
 date                     0
pm10                     3
pm2_5                    3
carbon_monoxide          3
nitrogen_dioxide         3
sulphur_dioxide          3
ozone                    3
aerosol_optical_depth    3
dust                     3
uv_index                 3
us_aqi                   4
european_aqi             4
dtype: int64
Duplicate rows: 0


In [9]:
df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
df = df.dropna(subset=["date"]).drop_duplicates(subset="date")
df = df.sort_values("date").reset_index(drop=True)

# Drop rows where every measurement is empty (like the first 3 rows)
value_cols = [c for c in df.columns if c != "date"]
df = df.dropna(subset=value_cols, how="all")

# Fill small gaps using neighbouring days
df = df.set_index("date")
df[value_cols] = df[value_cols].interpolate(method="time", limit=3)
df = df.dropna(subset=["us_aqi"]).reset_index()

print("Remaining missing values:\n", df.isnull().sum())

Remaining missing values:
 date                     0
pm10                     0
pm2_5                    0
carbon_monoxide          0
nitrogen_dioxide         0
sulphur_dioxide          0
ozone                    0
aerosol_optical_depth    0
dust                     0
uv_index                 0
us_aqi                   0
european_aqi             0
dtype: int64


In [10]:
df["Year"] = df["date"].dt.year
df["Month"] = df["date"].dt.month
df["Month_Name"] = df["date"].dt.month_name()
df["Day_Name"] = df["date"].dt.day_name()

def season(m):
    if m in [3, 4, 5, 6]: return "Summer"
    if m in [7, 8, 9, 10]: return "Monsoon"
    return "Winter"
df["Season"] = df["Month"].map(season)

In [11]:
df["US_AQI_Category"] = pd.cut(
    df["us_aqi"],
    bins=[-1, 50, 100, 150, 200, 300, 1000],
    labels=["Good", "Moderate", "Unhealthy for Sensitive",
            "Unhealthy", "Very Unhealthy", "Hazardous"])

cat_counts = df["US_AQI_Category"].value_counts().reset_index()
cat_counts.columns = ["Category", "Days"]
print(cat_counts)

px.pie(cat_counts, names="Category", values="Days", hole=0.4,
       title="Distribution of Days by US AQI Category").show()

                  Category  Days
0                 Moderate   322
1  Unhealthy for Sensitive   140
2                Unhealthy    34
3                     Good    17
4           Very Unhealthy     0
5                Hazardous     0


In [12]:
df["AQI_7day_avg"] = df["us_aqi"].rolling(7).mean()
df["AQI_30day_avg"] = df["us_aqi"].rolling(30).mean()

px.line(df, x="date", y=["us_aqi", "AQI_7day_avg", "AQI_30day_avg"],
        title="US AQI Trend with Moving Averages").show()

px.line(df, x="date", y=["us_aqi", "european_aqi"],
        title="US AQI vs European AQI").show()

In [14]:
monthly = df.groupby(df["date"].dt.to_period("M"))["us_aqi"].mean().reset_index()
monthly["date"] = monthly["date"].astype(str)
px.line(monthly, x="date", y="us_aqi", markers=True,
        title="Monthly Average US AQI").show()

In [16]:
by_season = df.groupby("Season")["us_aqi"].mean().round(1).reset_index()
print(by_season)
px.bar(by_season, x="Season", y="us_aqi", text="us_aqi",
       title="Average AQI by Season").show()

    Season  us_aqi
0  Monsoon    92.8
1   Summer    88.9
2   Winter    90.8


In [17]:
pollutants = ["pm10", "pm2_5", "carbon_monoxide", "nitrogen_dioxide",
              "sulphur_dioxide", "ozone", "dust"]

print(df[pollutants].mean().round(2))

corr = df[pollutants + ["aerosol_optical_depth", "uv_index", "us_aqi"]].corr().round(2)
px.imshow(corr, text_auto=".2f", title="Correlation Heatmap").show()
print(corr["us_aqi"].sort_values(ascending=False))

px.scatter(df, x="pm2_5", y="us_aqi", trendline="ols",
           title="PM2.5 vs US AQI").show()

pm10                 49.75
pm2_5                30.35
carbon_monoxide     395.63
nitrogen_dioxide     21.06
sulphur_dioxide      11.03
ozone                68.26
dust                 23.59
dtype: float64


us_aqi                   1.00
pm2_5                    0.95
carbon_monoxide          0.79
pm10                     0.71
sulphur_dioxide          0.69
nitrogen_dioxide         0.65
ozone                    0.29
aerosol_optical_depth    0.17
dust                    -0.33
uv_index                -0.42
Name: us_aqi, dtype: float64


In [18]:
print("Top 10 most polluted days:")
print(df.nlargest(10, "us_aqi")[["date", "us_aqi", "pm2_5", "pm10"]])

print("Top 10 cleanest days:")
print(df.nsmallest(10, "us_aqi")[["date", "us_aqi", "pm2_5", "pm10"]])

Top 10 most polluted days:
          date      us_aqi      pm2_5        pm10
490 2026-01-02  167.416667  80.308333   82.054167
35  2022-08-11  162.250000  68.050000   99.004167
356 2025-01-12  157.333333  70.258333   71.008333
11  2022-03-12  156.791667  63.487500   91.100000
177 2023-11-01  156.583333  58.083333   84.787500
213 2024-02-01  156.125000  61.729167   94.016667
492 2026-02-02  155.375000  58.795833   60.754167
474 2025-11-10  155.333333  56.333333   73.808333
494 2026-03-02  155.291667  65.295833   66.916667
30  2022-07-11  155.166667  76.358333  111.645833
Top 10 cleanest days:
          date     us_aqi      pm2_5       pm10
74  2023-02-06  33.041667   9.587500  20.812500
182 2023-11-06  39.250000  11.462500  20.462500
170 2023-10-06  39.416667   7.737500  12.237500
207 2024-01-07  40.958333   8.904167  13.112500
411 2025-06-07  42.625000  16.791667  27.154167
62  2023-01-06  44.958333   8.320833  15.062500
232 2024-03-08  45.458333  16.279167  39.504167
349 2025-01-05  4